# Document Processing Prototype

This notebook provides an incremental approach to testing the document processing pipeline for construction specifications.

## Overview

The notebook is structured to allow step-by-step development and validation:
1. **Environment Setup** - Install dependencies and configure paths
2. **Configuration** - Set up Docling pipeline options
3. **File Loading** - Load and inspect PDF files from the data directory
4. **Document Parsing** - Extract content using Docling
5. **Results Visualization** - Display and export parsed results
6. **Future Phases** - Placeholders for CSI classification, fact extraction, etc.



In [1]:
# Cell 1: Environment Setup and Dependencies

# Install required packages (run this cell first)
# Uncomment the following lines to install dependencies
# !pip install docling docling-core pymupdf pillow pandas numpy matplotlib IPython

import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional

# Configure logging for debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set up paths
PROJECT_ROOT = Path.cwd().parent  # Go up one level from notebooks/
DATA_DIR = PROJECT_ROOT / "data"

# List available PDF files
pdf_files = list(DATA_DIR.glob("*.pdf"))
print(f"\nFound {len(pdf_files)} PDF files:")
for pdf_file in pdf_files:
    print(f"  - {pdf_file.name}")



Found 3 PDF files:
  - Submittal and Product Description_redacted.pdf
  - Architectural Drawings_redacted.pdf
  - Spec 14 24 00 - Hydraulic Elevators.pdf


### Document Processing Configuration

In [ ]:
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption

# Configure Docling pipeline options for construction documents
def create_docling_config():
    """Create Docling configuration optimized for construction documents"""
    
    # PDF pipeline options based on real construction document analysis
    pdf_options = PdfPipelineOptions(
        # Large format drawing support (up to 4000 points + margin)
        max_page_width=4000,
        max_page_height=4000,
        
        # Annotation extraction (for architectural drawings)
        extract_annotations=True,
        annotation_types=["text", "dimension", "callout", "symbol"],
        
        # XObject handling (for embedded drawings like BBA, BBA1, etc.)
        extract_xobjects=True,
        xobject_types=["BBA", "BBA1", "BBA2", "BBA3", "BBA4", "BBA5"],
        
        # Page rotation handling
        detect_rotation=True,
        rotation_angles=[0, 90, 180, 270],
        
        # OCR settings
        ocr_settings={
            "enable_ocr": True,
            "ocr_language": "eng",
            "ocr_confidence_threshold": 0.8
        },
        
        # Table extraction
        table_settings={
            "extract_tables": True,
            "table_detection_confidence": 0.7
        }
    )
    
    # Format options
    format_options = {
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
    }
    
    return format_options

# Initialize document converter
try:
    format_options = create_docling_config()
    converter = DocumentConverter(format_options=format_options)
    print("✅ Docling document converter initialized successfully")
    print(f"   - Max page dimensions: 4000x4000 points")
    print(f"   - Annotation extraction: Enabled")
    print(f"   - XObject extraction: Enabled")
    print(f"   - OCR: Enabled")
    print(f"   - Table extraction: Enabled")
except Exception as e:
    print(f"❌ Error initializing Docling converter: {e}")
    print("Make sure to install docling and docling-core packages")
    converter = None


✅ Docling document converter initialized successfully
   - Max page dimensions: 4000x4000 points
   - Annotation extraction: Enabled
   - XObject extraction: Enabled
   - OCR: Enabled
   - Table extraction: Enabled


### File System Loading and Metadata Extraction

In [ ]:
import fitz  # PyMuPDF for basic PDF metadata
from pathlib import Path

def get_pdf_metadata(pdf_path: Path) -> Dict[str, Any]:
    """Extract basic metadata from PDF file"""
    try:
        doc = fitz.open(pdf_path)
        metadata = {
            "filename": pdf_path.name,
            "file_size_bytes": pdf_path.stat().st_size,
            "page_count": len(doc),
            "pdf_version": doc.metadata.get("format", "Unknown"),
            "creator": doc.metadata.get("creator", "Unknown"),
            "producer": doc.metadata.get("producer", "Unknown"),
            "creation_date": doc.metadata.get("creationDate", ""),
            "modification_date": doc.metadata.get("modDate", ""),
            "title": doc.metadata.get("title", ""),
            "subject": doc.metadata.get("subject", ""),
        }
        
        # Get first page dimensions for drawing documents
        if len(doc) > 0:
            page = doc[0]
            rect = page.rect
            metadata["first_page_dimensions"] = {
                "width": rect.width,
                "height": rect.height,
                "rotation": page.rotation
            }
            
            # Count annotations on first page
            try:
                annotations = page.annots()
                metadata["first_page_annotations"] = len(annotations)
            except Exception as e:
                logger.warning(f"Could not get annotations for {pdf_path}: {e}")
                metadata["first_page_annotations"] = 0
        
        doc.close()
        return metadata
    except Exception as e:
        logger.error(f"Error reading metadata for {pdf_path}: {e}")
        return {"filename": pdf_path.name, "error": str(e)}

def display_document_info(pdf_files: List[Path]) -> Dict[str, Dict[str, Any]]:
    """Display information about all PDF files"""
    print("📄 Document Information")
    print("=" * 80)
    
    document_info = {}
    
    for pdf_file in pdf_files:
        print(f"\n📋 {pdf_file.name}")
        print("-" * 60)
        
        metadata = get_pdf_metadata(pdf_file)
        document_info[pdf_file.name] = metadata
        
        if "error" in metadata:
            print(f"❌ Error: {metadata['error']}")
            continue
            
        # File information
        size_mb = metadata["file_size_bytes"] / (1024 * 1024)
        print(f"   File size: {size_mb:.2f} MB")
        print(f"   Pages: {metadata['page_count']}")
        print(f"   PDF version: {metadata['pdf_version']}")
        
        # Creator information
        print(f"   Creator: {metadata['creator']}")
        print(f"   Producer: {metadata['producer']}")
        
        # Document type inference
        doc_type = infer_document_type(pdf_file.name, metadata)
        print(f"   Document type: {doc_type}")
        
        # Page information for drawings
        if "first_page_dimensions" in metadata:
            dims = metadata["first_page_dimensions"]
            print(f"   First page: {dims['width']:.0f} x {dims['height']:.0f} points")
            if dims["rotation"] != 0:
                print(f"   Page rotation: {dims['rotation']}°")
            print(f"   Annotations on first page: {metadata.get('first_page_annotations', 0)}")
    
    return document_info

def infer_document_type(filename: str, metadata: Dict[str, Any]) -> str:
    """Infer document type from filename and metadata"""
    filename_lower = filename.lower()
    
    if "spec" in filename_lower and any(div in filename for div in ["14 24", "07 21", "03 30"]):
        return "CSI Specification"
    elif "drawing" in filename_lower or "plan" in filename_lower:
        return "Architectural Drawing"
    elif "sd" in filename_lower or "pd" in filename_lower:
        return "Submittal Document"
    else:
        return "Construction Document"

# Load and display information about all PDF files
if pdf_files:
    document_info = display_document_info(pdf_files)
    print(f"\n✅ Loaded information for {len(document_info)} documents")
else:
    print("❌ No PDF files found in data directory")
    document_info = {}


📄 Document Information

📋 Submittal and Product Description_redacted.pdf
------------------------------------------------------------


2025-10-16 15:42:02,997 - __main__ - ERROR - Error reading metadata for /Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/data/Submittal and Product Description_redacted.pdf: 'Page' object has no attribute 'get_annotations'


❌ Error: 'Page' object has no attribute 'get_annotations'

📋 Architectural Drawings_redacted.pdf
------------------------------------------------------------


2025-10-16 15:42:06,511 - __main__ - ERROR - Error reading metadata for /Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/data/Architectural Drawings_redacted.pdf: 'Page' object has no attribute 'get_annotations'


❌ Error: 'Page' object has no attribute 'get_annotations'

📋 Spec 14 24 00 - Hydraulic Elevators.pdf
------------------------------------------------------------


2025-10-16 15:42:07,558 - __main__ - ERROR - Error reading metadata for /Users/rsoares/dev/github/rafaeltuelho/construction-spec-assistant/data/Spec 14 24 00 - Hydraulic Elevators.pdf: 'Page' object has no attribute 'get_annotations'


❌ Error: 'Page' object has no attribute 'get_annotations'

✅ Loaded information for 3 documents


In [ ]:
# Cell 4: Document Parsing with Docling

def parse_document_with_docling(pdf_path: Path, document_name: str) -> Dict[str, Any]:
    """Parse a single PDF document using Docling"""
    if converter is None:
        raise ValueError("Docling converter not initialized. Run Cell 2 first.")
    
    print(f"🔄 Parsing {document_name}...")
    start_time = datetime.now()
    
    try:
        # Convert document
        result = converter.convert(str(pdf_path))
        doc = result.document
        
        # Extract basic document information
        doc_info = {
            "filename": pdf_path.name,
            "document_name": document_name,
            "parse_timestamp": start_time.isoformat(),
            "processing_time_seconds": (datetime.now() - start_time).total_seconds(),
            "success": True,
            "error": None
        }
        
        # Extract text content
        doc_info["full_text"] = doc.export_to_markdown()
        
        # Extract document structure
        doc_info["structure"] = []
        if hasattr(doc, 'chunks'):
            for chunk in doc.chunks:
                chunk_info = {
                    "type": chunk.label,
                    "text": chunk.text[:200] + "..." if len(chunk.text) > 200 else chunk.text,
                    "bbox": chunk.bbox if hasattr(chunk, 'bbox') else None
                }
                doc_info["structure"].append(chunk_info)
        
        # Extract tables
        doc_info["tables"] = []
        if hasattr(doc, 'tables'):
            for i, table in enumerate(doc.tables):
                table_info = {
                    "table_id": i,
                    "bbox": table.bbox if hasattr(table, 'bbox') else None,
                    "row_count": len(table.cells) if hasattr(table, 'cells') else 0,
                    "preview": str(table)[:200] + "..." if len(str(table)) > 200 else str(table)
                }
                doc_info["tables"].append(table_info)
        
        # Extract figures/images
        doc_info["figures"] = []
        if hasattr(doc, 'figures'):
            for i, figure in enumerate(doc.figures):
                figure_info = {
                    "figure_id": i,
                    "bbox": figure.bbox if hasattr(figure, 'bbox') else None,
                    "caption": figure.caption if hasattr(figure, 'caption') else None
                }
                doc_info["figures"].append(figure_info)
        
        # Extract annotations (if available)
        doc_info["annotations"] = []
        if hasattr(doc, 'annotations'):
            for i, annotation in enumerate(doc.annotations):
                annotation_info = {
                    "annotation_id": i,
                    "type": annotation.label if hasattr(annotation, 'label') else "unknown",
                    "text": annotation.text if hasattr(annotation, 'text') else None,
                    "bbox": annotation.bbox if hasattr(annotation, 'bbox') else None
                }
                doc_info["annotations"].append(annotation_info)
        
        print(f"✅ Successfully parsed {document_name}")
        print(f"   - Processing time: {doc_info['processing_time_seconds']:.2f} seconds")
        print(f"   - Text length: {len(doc_info['full_text'])} characters")
        print(f"   - Structure elements: {len(doc_info['structure'])}")
        print(f"   - Tables: {len(doc_info['tables'])}")
        print(f"   - Figures: {len(doc_info['figures'])}")
        print(f"   - Annotations: {len(doc_info['annotations'])}")
        
        return doc_info
        
    except Exception as e:
        error_info = {
            "filename": pdf_path.name,
            "document_name": document_name,
            "parse_timestamp": start_time.isoformat(),
            "processing_time_seconds": (datetime.now() - start_time).total_seconds(),
            "success": False,
            "error": str(e),
            "full_text": None,
            "structure": [],
            "tables": [],
            "figures": [],
            "annotations": []
        }
        
        print(f"❌ Error parsing {document_name}: {e}")
        return error_info

# Parse all documents
parsed_documents = {}

if converter is not None and pdf_files:
    print("🚀 Starting document parsing...")
    print("=" * 80)
    
    for pdf_file in pdf_files:
        document_name = pdf_file.stem  # filename without extension
        parsed_doc = parse_document_with_docling(pdf_file, document_name)
        parsed_documents[document_name] = parsed_doc
        print()  # Add spacing between documents
    
    print(f"✅ Completed parsing {len(parsed_documents)} documents")
else:
    print("❌ Cannot parse documents. Make sure:")
    print("   1. Docling converter is initialized (run Cell 2)")
    print("   2. PDF files are available (run Cell 1)")


In [ ]:
# Cell 5: Parsing Results Visualization and Export

import json
from IPython.display import display, HTML, Markdown

def display_parsing_summary(parsed_documents: Dict[str, Dict[str, Any]]):
    """Display a summary of parsing results"""
    print("📊 Parsing Results Summary")
    print("=" * 80)
    
    total_docs = len(parsed_documents)
    successful_docs = sum(1 for doc in parsed_documents.values() if doc.get('success', False))
    failed_docs = total_docs - successful_docs
    
    print(f"Total documents: {total_docs}")
    print(f"Successfully parsed: {successful_docs}")
    print(f"Failed: {failed_docs}")
    print()
    
    # Document details
    for doc_name, doc_info in parsed_documents.items():
        status = "✅" if doc_info.get('success', False) else "❌"
        print(f"{status} {doc_name}")
        
        if doc_info.get('success', False):
            print(f"   Processing time: {doc_info.get('processing_time_seconds', 0):.2f}s")
            print(f"   Text length: {len(doc_info.get('full_text', '')):,} chars")
            print(f"   Structure elements: {len(doc_info.get('structure', []))}")
            print(f"   Tables: {len(doc_info.get('tables', []))}")
            print(f"   Figures: {len(doc_info.get('figures', []))}")
            print(f"   Annotations: {len(doc_info.get('annotations', []))}")
        else:
            print(f"   Error: {doc_info.get('error', 'Unknown error')}")
        print()

def display_document_structure(doc_name: str, doc_info: Dict[str, Any]):
    """Display the structure of a parsed document"""
    if not doc_info.get('success', False):
        print(f"❌ Cannot display structure for {doc_name}: {doc_info.get('error')}")
        return
    
    print(f"📋 Document Structure: {doc_name}")
    print("-" * 60)
    
    structure = doc_info.get('structure', [])
    if structure:
        for i, chunk in enumerate(structure[:10]):  # Show first 10 elements
            chunk_type = chunk.get('type', 'unknown')
            text_preview = chunk.get('text', '')[:100]
            print(f"{i+1:2d}. [{chunk_type:15s}] {text_preview}")
        
        if len(structure) > 10:
            print(f"    ... and {len(structure) - 10} more elements")
    else:
        print("No structure elements found")
    print()

def display_text_preview(doc_name: str, doc_info: Dict[str, Any], max_chars: int = 500):
    """Display a preview of the extracted text"""
    if not doc_info.get('success', False):
        print(f"❌ Cannot display text for {doc_name}: {doc_info.get('error')}")
        return
    
    print(f"📝 Text Preview: {doc_name}")
    print("-" * 60)
    
    full_text = doc_info.get('full_text', '')
    if full_text:
        preview = full_text[:max_chars]
        print(preview)
        if len(full_text) > max_chars:
            print(f"\n... ({len(full_text) - max_chars:,} more characters)")
    else:
        print("No text content extracted")
    print()

def save_parsing_results(parsed_documents: Dict[str, Dict[str, Any]], output_dir: Path):
    """Save parsing results to JSON files"""
    output_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save individual documents
    for doc_name, doc_info in parsed_documents.items():
        filename = f"{doc_name}_parsed_{timestamp}.json"
        filepath = output_dir / filename
        
        # Remove full_text from individual files to keep them manageable
        save_info = doc_info.copy()
        if 'full_text' in save_info:
            save_info['text_length'] = len(save_info['full_text'])
            del save_info['full_text']
        
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(save_info, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Saved {doc_name} results to {filepath}")
    
    # Save summary file
    summary_file = output_dir / f"parsing_summary_{timestamp}.json"
    summary = {
        "timestamp": timestamp,
        "total_documents": len(parsed_documents),
        "successful_documents": sum(1 for doc in parsed_documents.values() if doc.get('success', False)),
        "document_names": list(parsed_documents.keys()),
        "processing_times": {
            doc_name: doc_info.get('processing_time_seconds', 0) 
            for doc_name, doc_info in parsed_documents.items()
        }
    }
    
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Saved summary to {summary_file}")

# Display parsing results
if parsed_documents:
    display_parsing_summary(parsed_documents)
    
    # Show structure for first successful document
    for doc_name, doc_info in parsed_documents.items():
        if doc_info.get('success', False):
            display_document_structure(doc_name, doc_info)
            display_text_preview(doc_name, doc_info)
            break
    
    # Save results to JSON files
    output_dir = NOTEBOOKS_DIR / "parsing_results"
    save_parsing_results(parsed_documents, output_dir)
    
    print("✅ Parsing results displayed and saved")
else:
    print("❌ No parsed documents to display. Run Cell 4 first.")


## Next Phase: CSI Division Classification

This cell will contain the CSI (Construction Specifications Institute) division classification logic. Based on the parsed document content, we'll:

1. **Identify CSI Division Codes** - Extract codes like "14 24 00" from document content
2. **Classify Document Sections** - Map content to appropriate CSI divisions
3. **Validate CSI Format** - Ensure codes follow the standard format
4. **Extract Division Metadata** - Gather additional context about each division

### Expected Implementation:
- CSI pattern matching using regex
- Division code validation
- Document section classification
- Metadata extraction for each identified division

*Add your CSI classification code here when ready to implement this phase.*


## Next Phase: Fact Extraction

This cell will contain the fact extraction logic to identify and extract structured facts from the parsed documents. We'll focus on:

1. **Specification Facts** - Extract requirements, specifications, and constraints
2. **Numerical Facts** - Identify measurements, quantities, and performance values
3. **Product Information** - Extract manufacturer, model, and product details
4. **Standards and Codes** - Identify referenced standards and building codes

### Expected Implementation:
- Pattern matching for specification text
- Unit normalization (inches, feet, pounds, etc.)
- Operator detection (>=, <=, =, etc.)
- Fact validation and confidence scoring
- JSON output with structured fact format

### Example Fact Output:
```json
{
  "topic": "hydraulic_elevator",
  "attribute": "capacity_lbs",
  "operator": "=",
  "value": 2500,
  "unit": "lbs",
  "source": {
    "pdf": "Spec_14_24_00_Hydraulic_Elevators.pdf",
    "page": 2,
    "span": "Part 2, Section 2.1.A"
  }
}
```

*Add your fact extraction code here when ready to implement this phase.*


## Next Phase: Passage Chunking and Indexing

This cell will contain the passage extraction and chunking logic to prepare documents for search and retrieval. We'll implement:

1. **Text Chunking** - Split documents into meaningful passages
2. **Metadata Preservation** - Maintain source information for each chunk
3. **Search Preparation** - Format passages for vector and keyword search
4. **Quality Validation** - Ensure chunks are appropriately sized and meaningful

### Expected Implementation:
- Semantic chunking based on document structure
- Overlap handling for context preservation
- Metadata extraction for each passage
- Chunk size optimization (target: 200-500 tokens)
- Source citation preservation

### Expected Passage Format:
```json
{
  "passage_id": "spec_14_24_00_chunk_001",
  "text": "Hydraulic elevators shall have a minimum capacity of 2,500 pounds...",
  "source": {
    "document": "Spec_14_24_00_Hydraulic_Elevators.pdf",
    "page": 2,
    "section": "Part 2, Section 2.1.A",
    "csi_division": "14 24 00"
  },
  "metadata": {
    "chunk_size": 347,
    "contains_specifications": true,
    "contains_measurements": true
  }
}
```

*Add your passage chunking code here when ready to implement this phase.*


## Development Notes and Next Steps

### Current Status
✅ **Completed Phases:**
- Environment setup and dependency management
- Docling configuration for construction documents
- PDF file loading and metadata extraction
- Document parsing with Docling
- Results visualization and JSON export

### Next Development Phases
🔄 **Ready to Implement:**
- CSI Division Classification (Cell 6)
- Fact Extraction (Cell 7)
- Passage Chunking (Cell 8)

### Usage Instructions

1. **Start Here**: Run cells 1-5 in sequence to parse your documents
2. **Inspect Results**: Review the parsing results in Cell 5
3. **Add Next Phase**: Implement CSI classification in Cell 6
4. **Iterate**: Continue adding cells for each processing phase
5. **Validate**: Check results at each step before proceeding

### Tips for Development

- **Run cells incrementally** - Don't run all cells at once
- **Check outputs** - Verify parsing results before adding new phases
- **Save frequently** - Export results to JSON for inspection
- **Debug issues** - Use the logging output to troubleshoot problems
- **Extend gradually** - Add complexity one feature at a time

### Troubleshooting

- **Docling errors**: Make sure all dependencies are installed
- **File not found**: Verify the `data/` directory contains your PDF files
- **Memory issues**: Process documents one at a time for large files
- **Parsing failures**: Check PDF format compatibility with Docling

### Integration with Backend

Once you've validated the processing pipeline in this notebook, you can:
1. Extract the working code into Python modules
2. Integrate with the FastAPI backend
3. Add database persistence
4. Implement the full agentic workflow

*This notebook serves as your development sandbox for the document processing system.*
